In [2]:
# import core libraries
import asyncio
import os
import openai
from dotenv import load_dotenv

"""
 “RuntimeError: This event loop is already running”

The issue pops up in various environments, such as web servers, GUI applications and in Jupyter notebooks.

This module patches asyncio to allow nested use of asyncio.run and loop.run_until_complete.
"""
import nest_asyncio
nest_asyncio.apply()

# import aimakerspace libraries
from aimakerspace.text_utils import TextFileLoader, CharacterTextSplitter
from aimakerspace.vectordatabase import VectorDatabase

# load the data
text_loader = TextFileLoader("data/test-scan.txt")
documents = text_loader.load_documents()
split_documents = CharacterTextSplitter().split_texts(documents)

# get the API key from .env file instead of asking for it
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

In [4]:
vector_db = VectorDatabase()

In [6]:
vector_db.search_by_text("How many devices are there with the ssh service open?", k=3)

[]

In [4]:
from aimakerspace.openai_utils.prompts import (
    UserRolePrompt,
    SystemRolePrompt,
    AssistantRolePrompt,
)

from aimakerspace.openai_utils.chatmodel import ChatOpenAI

chat_openai = ChatOpenAI()
user_prompt_template = "{content}"
user_role_prompt = UserRolePrompt(user_prompt_template)
system_prompt_template = (
    "You are an expert in {expertise}, you always answer in a kind way."
)
system_role_prompt = SystemRolePrompt(system_prompt_template)

messages = [
    system_role_prompt.create_message(expertise="Cybersecurity"),
    user_role_prompt.create_message(
        content="What is the best way to scan a network for vulnerabilities?"
    ),
]

response = chat_openai.run(messages)

In [5]:
print(response)

Scanning a network for vulnerabilities is a crucial step in ensuring its security. Here are some best practices to consider when conducting a vulnerability scan:

1. **Define the Scope**: Before you start, clearly define what parts of the network you’ll be scanning. Identify all devices, servers, and applications that need to be assessed.

2. **Choose the Right Tools**: There are several excellent vulnerability scanning tools available. Some of the most popular ones include:
   - **Nessus**: Widely used and offers a comprehensive set of checks.
   - **Qualys**: A cloud-based solution that provides continuous monitoring.
   - **OpenVAS**: An open-source option with a robust set of features.
   - **Nmap**: Useful for network discovery and security auditing.

3. **Prioritize Scanning Frequency**: Regular scans are essential. Depending on your environment, you might choose to perform scans weekly, monthly, or quarterly. After significant changes in your network, consider a scan as well.

4

In [13]:
RAG_SYSTEM_TEMPLATE = """You are a knowledgeable assistant that answers questions based strictly on provided context.

Instructions:
- Only answer questions using information from the provided context
- If the context doesn't contain relevant information, respond with "I don't know"
- Be accurate and cite specific parts of the context when possible
- Keep responses {response_style} and {response_length}
- Only use the provided context. Do not use external knowledge.
- Only provide answers when you are confident the context supports your response."""

RAG_USER_TEMPLATE = """Context Information:
{context}

Number of relevant sources found: {context_count}
{similarity_scores}

Question: {user_query}

Please provide your answer based solely on the context above."""

rag_system_prompt = SystemRolePrompt(
    RAG_SYSTEM_TEMPLATE,
    strict=True,
    defaults={
        "response_style": "concise",
        "response_length": "brief"
    }
)

rag_user_prompt = UserRolePrompt(
    RAG_USER_TEMPLATE,
    strict=True,
    defaults={
        "context_count": "",
        "similarity_scores": ""
    }
)

In [33]:
class RetrievalAugmentedQAPipeline:
    def __init__(self, llm: ChatOpenAI(), vector_db_retriever: VectorDatabase, 
                 response_style: str = "detailed", include_scores: bool = False) -> None:
        self.llm = llm
        self.vector_db_retriever = vector_db_retriever
        self.response_style = response_style
        self.include_scores = include_scores

    def run_pipeline(self, user_query: str, k: int = 4, **system_kwargs) -> dict:
        # Retrieve relevant contexts
        context_list = self.vector_db_retriever.search_by_text(user_query, k=k)
        
        context_prompt = ""
        similarity_scores = []
        
        for i, (context, score, _) in enumerate(context_list, 1):
            context_prompt += f"[Source {i}]: {context}\n\n"
            similarity_scores.append(f"Source {i}: {score:.3f}")
        
        # Create system message with parameters
        system_params = {
            "response_style": self.response_style,
            "response_length": system_kwargs.get("response_length", "detailed")
        }
        
        formatted_system_prompt = rag_system_prompt.create_message(**system_params)
        
        user_params = {
            "user_query": user_query,
            "context": context_prompt.strip(),
            "context_count": len(context_list),
            "similarity_scores": f"Relevance scores: {', '.join(similarity_scores)}" if self.include_scores else ""
        }
        
        formatted_user_prompt = rag_user_prompt.create_message(**user_params)

        return {
            "response": self.llm.run([formatted_system_prompt, formatted_user_prompt]), 
            "context": context_list,
            "context_count": len(context_list),
            "similarity_scores": similarity_scores if self.include_scores else None,
            "prompts_used": {
                "system": formatted_system_prompt,
                "user": formatted_user_prompt
            }
        }

In [36]:
rag_pipeline = RetrievalAugmentedQAPipeline(
    vector_db_retriever=vector_db,
    llm=chat_openai,
    response_style="detailed",
    include_scores=True
)

result = rag_pipeline.run_pipeline(
    "How many devices with the ssh service open are there?",
    k=3,
    response_length="comprehensive", 
    include_warnings=True,
    confidence_required=True
)

print(f"Response: {result['response']}")
print(f"\nContext Count: {result['context_count']}")
print(f"Similarity Scores: {result['similarity_scores']}")


Response: Based on the provided context, the following devices have the SSH service open:

1. **Host: 190.7.193.249** - Cisco SSH 1.25 (protocol 1.99)
2. **Host: 190.7.193.66** - OpenSSH 5.5p1 Debian 6+squeeze2 (protocol 2.0)
3. **Host: 190.7.193.67** - OpenSSH 5.5p1 Debian 6+squeeze2 (protocol 2.0)
4. **Host: 190.7.193.70** - OpenSSH 6.0p1 Debian 4+deb7u3 (protocol 2.0)
5. **Host: 190.7.193.8** - Cisco SSH 1.25 (protocol 1.99)
6. **Host: 190.7.193.11** - Cisco SSH 1.25 (protocol 1.99)
7. **Host: 190.7.193.14** - Cisco SSH 1.25 (protocol 1.99)
8. **Host: 190.7.193.16** - Cisco SSH 1.25 (protocol 1.99)
9. **Host: 190.7.193.19** - Cisco SSH 1.25 (protocol 1.99)
10. **Host: 190.7.193.20** - Cisco SSH 1.25 (protocol 1.99)
11. **Host: 190.7.193.21** - Cisco SSH 1.25 (protocol 1.99)

In total, there are **11 devices with the SSH service open**.

Context Count: 3
Similarity Scores: ['Source 1: 0.561', 'Source 2: 0.552', 'Source 3: 0.538']
